# 04 学習

`01_data_fetch` が作った `data/processed/observations.parquet`（1 行 ＝ 都市 × 正時）から、雲底高度（シーリング）を予測するモデルを学習し、
`models/cloud_base_v1.joblib` と `reports/training.json` に保存する。

- 特徴量・目的変数・分割・学習・予測・評価関数の中身は `ml/` にある（テスト済み）。このノートブックは順に呼ぶだけ。
- 特徴量は Open-Meteo の値と、都市・時刻から計算できるものだけ（METAR の列は使わない。推論のときにないため）。
- モデルは 2 都市で 1 つの LightGBM。目的関数は MAE。設定は固定で、木の数だけを検証の年（2024）で決める。
- **学習の年を 6 年（2018〜2023）と 3 年（2021〜2023）で 1 回だけ比べる**（`docs/spec.md`「9. 分割と評価」）。
  比べるのは、検証の年の、2 都市を合わせた全体の MAE。3 年のほうが小さければ 3 年、同じか大きければ 6 年を採る。
- **テストの年（2025）は、読み込んだ直後に落とす**（ここでは見ない。`05_evaluate` で 1 回だけ使う）。
- 既定は **サンプル**（`tests/fixtures/data_raw/` の 2 日ぶん。動くことの確認用で、数値に意味はない。一時的な場所に書く）。
  コミットしてある成果物は、環境変数 `DATA_FETCH_MODE=full` を付けて全期間で実行したもの。外部とは通信しない。
- 時刻は UTC（分割の年も UTC）。単位は摂氏・メートル。

出典：Weather data by Open-Meteo.com（CC BY 4.0）、Iowa Environmental Mesonet（Iowa State University）。

In [ ]:
import json
import os
import sys
import tempfile
from pathlib import Path

import lightgbm as lgb
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from app.cities import City
from ml import (
    archive,
    baseline,
    dataset,
    evaluate,
    features,
    metar,
    model,
    raw_store,
    split,
    target,
    train,
)

## 0. 設定・読み込み

In [ ]:
MODE = os.environ.get("DATA_FETCH_MODE", "sample")

if MODE == "full":
    OUTPUT_ROOT = ROOT
elif MODE == "sample":
    OUTPUT_ROOT = Path(tempfile.mkdtemp())
else:
    raise ValueError("DATA_FETCH_MODE must be 'sample' or 'full'")

MODEL_PATH = OUTPUT_ROOT / "models" / "cloud_base_v1.joblib"
REPORT_PATH = OUTPUT_ROOT / "reports" / "training.json"
# 比べる 2 つの学習期間。
CANDIDATES = {
    "train_2018_2023": split.SPLIT_YEARS[split.TRAIN],
    "train_2021_2023": split.TRAIN_YEARS_SHORT,
}
VALIDATION_YEARS = split.SPLIT_YEARS[split.VALIDATION]

pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", "{:.2f}".format)
print(f"mode={MODE}", list(CANDIDATES))

In [ ]:
if MODE == "full":
    observations = dataset.load_dataset(ROOT / "data" / "processed" / "observations.parquet")
    dataset.validate_dataset(observations)
    # テストの年は、ここで落とす（学習でも比較でも見ない）。
    is_test = split.rows_in_years(observations, split.SPLIT_YEARS[split.TEST])
    observations = observations[~is_test].reset_index(drop=True)
else:
    raw_root = ROOT / "tests" / "fixtures" / "data_raw"
    parts = {}
    for city in City:
        weather_path = raw_store.raw_path(raw_root, "open_meteo_archive", city, 2025, "json")
        metar_path = raw_store.raw_path(raw_root, "iem", city, 2025, "csv")
        parts[city.value] = (
            archive.parse_archive(weather_path.read_bytes().decode("utf-8")),
            metar.parse_metar(metar_path.read_bytes().decode("utf-8")),
        )
    observations = dataset.build_dataset(parts)
    dataset.validate_dataset(observations)

print(observations.shape, observations["time"].min(), observations["time"].max())

## 1. 特徴量と目的変数

`target_m` が欠損の行は、学習・評価から除く行（`ml/train.py` と `ml/evaluate.py` が除く）。

In [ ]:
X = features.features(observations)
y = target.target_m(observations)

if MODE == "full":
    train_rows = {
        name: split.rows_in_years(observations, years) for name, years in CANDIDATES.items()
    }
    validation_rows = split.rows_in_years(observations, VALIDATION_YEARS)
else:
    # fixture は 2025 年の 2 日ぶんだけなので、分割で絞らず、同じ行で学習と評価をする。
    everything = pd.Series(True, index=observations.index)
    train_rows = {name: everything for name in CANDIDATES}
    validation_rows = everything

counts = pd.DataFrame(
    {
        **{
            name: y[rows].notna().groupby(observations["city"][rows]).sum()
            for name, rows in train_rows.items()
        },
        "validation": y[validation_rows]
        .notna()
        .groupby(observations["city"][validation_rows])
        .sum(),
    }
)
print(list(X.columns))
counts

## 2. 学習と、検証の年での評価

2 つの学習期間で、同じ特徴量・同じ設定・同じ打ち切りの決まりで学習する。
ベースライン（`03_baseline` と同じ 3 つ）も、検証の年で並べる。

In [ ]:
baselines = {
    "rule_of_thumb": baseline.baseline_m(
        observations["temperature_c"], observations["dew_point_c"]
    ),
    "always_cap": pd.Series(target.CEILING_CAP_M, index=observations.index, dtype="float64"),
    "rule_of_thumb_metar": baseline.baseline_m(
        observations["metar_temperature_c"], observations["metar_dew_point_c"]
    ),
}


def report_for(rows, label, prediction):
    """`rows` の行の指標（全体・都市別・都市 × 季節別）。"""
    truth = y[rows]
    if label == "rule_of_thumb_metar":
        # 参考の予測だけ、予測が作れない行を評価から除く。
        truth = truth.where(prediction[rows].notna())
    return evaluate.report(observations[rows], truth, prediction[rows])


models = {}
candidates = {}
validation_mae = {}
for name, years in CANDIDATES.items():
    rows = train_rows[name]
    booster = train.fit(X[rows], y[rows], X[validation_rows], y[validation_rows])
    models[name] = model.CloudBaseModel(
        booster=booster, train_years=tuple(years), model_version=model.MODEL_VERSION_V1
    )
    predicted = model.predict_m(models[name], X)
    # 比較に使う値（丸める前）。
    validation_mae[name] = evaluate.mae(y[validation_rows], predicted[validation_rows])
    candidates[name] = {
        "train_years": list(years),
        "n_train": int(y[rows].notna().sum()),
        "best_iteration": int(booster.best_iteration),
        "validation": report_for(validation_rows, "model", predicted),
    }

validation_baselines = {
    label: report_for(validation_rows, label, prediction) for label, prediction in baselines.items()
}
pd.DataFrame(
    [
        {"prediction": label, "city": city, **metrics}
        for label, reports in [
            *[(name, candidate["validation"]) for name, candidate in candidates.items()],
            *validation_baselines.items(),
        ]
        for city, metrics in reports["by_city"].items()
    ]
)

## 3. 6 年と 3 年の比較

検証の年の、2 都市を合わせた全体の MAE で決める。3 年のほうが小さければ 3 年、同じか大きければ 6 年。

In [ ]:
long_name, short_name = list(CANDIDATES)
selected = short_name if validation_mae[short_name] < validation_mae[long_name] else long_name

print({name: round(value, 2) for name, value in validation_mae.items()})
print("selected:", selected)
pd.DataFrame(
    [
        {
            "candidate": name,
            "n_train": candidate["n_train"],
            "best_iteration": candidate["best_iteration"],
            **candidate["validation"]["overall"],
        }
        for name, candidate in candidates.items()
    ]
)

In [ ]:
# 特徴量の重要度（その特徴量で分けたときに、誤差が減った量の合計）。
gain = models[selected].booster.feature_importance(importance_type="gain")
feature_importance = {
    name: round(float(value), 2) for name, value in zip(features.FEATURE_NAMES, gain, strict=True)
}
pd.Series(feature_importance).sort_values(ascending=False)

## 4. 保存

採ったほうのモデルを保存する。実行した時刻は入れない（再実行しても同じファイルになる）。

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
model.save(models[selected], MODEL_PATH)

# 読み直して、同じ予測になることを確かめる。
loaded = model.load(MODEL_PATH)
pd.testing.assert_series_equal(model.predict_m(loaded, X), model.predict_m(models[selected], X))
print(MODEL_PATH.name, MODEL_PATH.stat().st_size, loaded.model_version, loaded.train_years)

In [ ]:
report = {
    "mode": MODE,
    "model_version": model.MODEL_VERSION_V1,
    "cap_m": target.CEILING_CAP_M,
    "feature_names": list(features.FEATURE_NAMES),
    "params": train.PARAMS,
    "max_rounds": train.MAX_ROUNDS,
    "early_stopping_rounds": train.EARLY_STOPPING_ROUNDS,
    "lightgbm_version": lgb.__version__,
    "validation_years": list(VALIDATION_YEARS),
    "selection_rule": "smaller overall validation MAE; the longer period on a tie",
    "selected": selected,
    "candidates": candidates,
    "validation_baselines": validation_baselines,
    "feature_importance": feature_importance,
}

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
text = json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False) + "\n"
REPORT_PATH.write_text(text, encoding="utf-8", newline="\n")
# 書いたファイルを読み直して確かめるだけ（入力の検証ではない）。
assert json.loads(REPORT_PATH.read_text(encoding="utf-8")) == report  # noqa: S101
print(REPORT_PATH.name, len(text))